# TractionMachinePM2D — Design Report

**48-slot, 8-pole, 3-phase Inset Permanent Magnet Traction Motor**  
Stator OD = 220 mm, Active length = 170 mm, n = 1500 rpm

---
_Generated from 2D FEM transient analysis (ElmerFEM). All results scaled to the full machine._

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms
import math
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# =============================================================
# MACHINE PARAMETERS
# =============================================================

# Topology
Qs = 48           # total slots
Qp = 8            # poles
PP = 4            # pole pairs
m  = 3            # phases
q  = 2            # slots/pole/phase
ns = 6            # slots per 45-degree sector

# Stator geometry [mm]
R_so = 110.0      # stator outer radius [mm]
R_si = 74.0       # stator bore [mm]
g    = 0.75       # air gap [mm]
R_sb = 73.625     # sliding boundary radius [mm]
R_ro = 73.25      # rotor outer [mm]
R_ri = 25.0       # shaft outer [mm]

# Magnet geometry [mm / deg]
mag_frac  = 0.80           # magnet arc fraction of pole pitch
h_m       = 4.0            # magnet radial depth [mm]
R_mi      = 69.25          # magnet inner radius [mm]
theta_s   = 45.0           # sector angle [deg]
theta_ml  = 4.5            # magnet left edge in sector [deg]
theta_mr  = 40.5           # magnet right edge in sector [deg]

# Slot / winding geometry [mm]
b1, h1, h2 = 2.5, 0.5, 0.5   # slot opening width, heights [mm]
b4, b5     = 5.0, 8.5         # winding bottom/top width [mm]
h5         = 20.0             # winding height [mm]
R_wb       = 74.989           # winding bottom radius [mm]
R_st       = 94.989           # slot top radius [mm]

# Magnet properties
H_PM  = 920000                                      # coercivity [A/m]
mu_r  = 1.03                                        # relative permeability
B_r   = 4 * 3.14159265e-7 * mu_r * H_PM            # remanence [T] ~1.192 T

# Electrical / speed
rpm  = 1500
fme  = rpm / 60               # mechanical frequency 25 Hz
fel  = fme * PP               # electrical frequency 100 Hz
dt   = 1 / (fme * 120)        # timestep [s]

# Loading (placeholder)
Is    = 200.0                 # peak ampere-turns per coil side [A]
Carea = 6.85e-5               # slot conductor area [m^2]

# Scaling
L_active  = 0.170             # active length [m]
N_sectors = 2 * PP            # = 8
SCALE     = N_sectors * L_active   # = 1.36 m, full-machine scaling

# Material densities [kg/m^3]
rho_fe  = 7650    # M350-50A steel
rho_cu  = 8900    # copper
rho_pm  = 7500    # NdFeB

print('Parameters loaded successfully.')
print(f'B_r = {B_r:.4f} T')
print(f'SCALE = {N_sectors} sectors x {L_active*1000:.0f} mm = {SCALE:.3f} m')
print(f'dt = {dt*1000:.4f} ms  ->  {int(round(1/(fme*dt)))} steps/mech.revolution')

Parameters loaded successfully.
B_r = 1.1908 T
SCALE = 8 sectors x 170 mm = 1.360 m
dt = 0.3333 ms  ->  120 steps/mech.revolution


## 1. Machine Parameters

In [2]:
# ─────────────────────────────────────────────────────────────
#  Formatted parameter table
# ─────────────────────────────────────────────────────────────
SEP = '-' * 55

print('=' * 55)
print('  TractionMachinePM2D — Machine Parameter Summary')
print('=' * 55)

print('\n--- TOPOLOGY ---')
print(SEP)
print(f'  {"Total slots  Qs":<30} {Qs}')
print(f'  {"Poles  Qp":<30} {Qp}')
print(f'  {"Pole pairs  PP":<30} {PP}')
print(f'  {"Phases  m":<30} {m}')
print(f'  {"Slots/pole/phase  q":<30} {q}')
print(f'  {"Slots per 45-deg sector  ns":<30} {ns}')

print('\n--- GEOMETRY ---')
print(SEP)
print(f'  {"Stator OD  2*R_so":<30} {2*R_so:.1f} mm')
print(f'  {"Stator bore  2*R_si":<30} {2*R_si:.1f} mm')
print(f'  {"Rotor OD  2*R_ro":<30} {2*R_ro:.2f} mm')
print(f'  {"Shaft OD  2*R_ri":<30} {2*R_ri:.1f} mm')
print(f'  {"Air gap  g":<30} {g:.2f} mm')
print(f'  {"Sliding boundary  R_sb":<30} {R_sb:.3f} mm')
print(f'  {"Active length  L_active":<30} {L_active*1000:.1f} mm')

print('\n--- MAGNET ---')
print(SEP)
print(f'  {"Radial depth  h_m":<30} {h_m:.1f} mm')
print(f'  {"Inner radius  R_mi":<30} {R_mi:.2f} mm')
print(f'  {"Arc fraction  mag_frac":<30} {mag_frac:.2f}  ({mag_frac*theta_s:.1f} deg of {theta_s:.0f} deg)')
print(f'  {"Remanence  B_r":<30} {B_r:.4f} T')
print(f'  {"Relative permeability  mu_r":<30} {mu_r:.3f}')
print(f'  {"Coercivity  H_PM":<30} {H_PM/1e3:.0f} kA/m')

print('\n--- SPEED & FREQUENCY ---')
print(SEP)
print(f'  {"Speed  n":<30} {rpm} rpm')
print(f'  {"Mechanical freq  fme":<30} {fme:.1f} Hz')
print(f'  {"Electrical freq  fel":<30} {fel:.1f} Hz')
print(f'  {"Time step  dt":<30} {dt*1000:.4f} ms')

print('\n--- LOADING (PLACEHOLDER) ---')
print(SEP)
print(f'  {"Peak coil current  Is":<30} {Is:.0f} A')
print(f'  {"Conductor area  Carea":<30} {Carea*1e6:.2f} mm^2')
print(f'  {"Peak current density  J":<30} {Is/Carea/1e6:.2f} MA/m^2')
print(f'  {"Scaling factor  SCALE":<30} {SCALE:.3f} m')
print('=' * 55)

  TractionMachinePM2D — Machine Parameter Summary

--- TOPOLOGY ---
-------------------------------------------------------
  Total slots  Qs                48
  Poles  Qp                      8
  Pole pairs  PP                 4
  Phases  m                      3
  Slots/pole/phase  q            2
  Slots per 45-deg sector  ns    6

--- GEOMETRY ---
-------------------------------------------------------
  Stator OD  2*R_so              220.0 mm
  Stator bore  2*R_si            148.0 mm
  Rotor OD  2*R_ro               146.50 mm
  Shaft OD  2*R_ri               50.0 mm
  Air gap  g                     0.75 mm
  Sliding boundary  R_sb         73.625 mm
  Active length  L_active        170.0 mm

--- MAGNET ---
-------------------------------------------------------
  Radial depth  h_m              4.0 mm
  Inner radius  R_mi             69.25 mm
  Arc fraction  mag_frac         0.80  (36.0 deg of 45 deg)
  Remanence  B_r                 1.1908 T
  Relative permeability  mu_r    1.030
  

## 2. Winding Analysis

In [3]:
# ─────────────────────────────────────────────────────────────
#  Winding factor calculation
# ─────────────────────────────────────────────────────────────
alpha_e = PP * 360.0 / Qs          # electrical angle between adjacent slots [deg]

# Distribution factor (q=2 slots/pole/phase, m=3)
kd = (math.sin(q * math.radians(alpha_e) / 2.0)
      / (q * math.sin(math.radians(alpha_e) / 2.0)))

# Pitch factor — full pitch (coil span = pole pitch = Qs/(2*PP) = 6 slots)
coil_span_slots = Qs // (2 * PP)   # = 6
# For full pitch, coil_span_slots == Qs/(2*PP), so ratio = 1 and kp = sin(pi/2) = 1
kp = math.sin(math.pi / 2.0 * coil_span_slots / (Qs / (2 * PP)))

kw = kd * kp

print('Winding Factor Analysis')
print('=' * 50)
print(f'  Electrical angle per slot:  alpha_e = {alpha_e:.1f} deg')
print(f'  Coil span:  {coil_span_slots} slots = {coil_span_slots * 360.0 / Qs:.1f} deg mechanical'
      f' = one pole pitch (full pitch)')
print()
print(f'  Distribution factor  kd = {kd:.4f}')
print(f'  Pitch factor         kp = {kp:.4f}  (full pitch -> kp=1)')
print(f'  Winding factor       kw = kd * kp = {kw:.4f}')
print()
print(f'  Note: kd < 1 because the {q} coils per phase group span {(q-1)*alpha_e:.0f} deg electrically')
print(f'  Fundamental winding factor kw = {kw:.4f} (typical: 0.90-0.97 for q=2)')

Winding Factor Analysis
  Electrical angle per slot:  alpha_e = 30.0 deg
  Coil span:  6 slots = 45.0 deg mechanical = one pole pitch (full pitch)

  Distribution factor  kd = 0.9659
  Pitch factor         kp = 1.0000  (full pitch -> kp=1)
  Winding factor       kw = kd * kp = 0.9659

  Note: kd < 1 because the 2 coils per phase group span 30 deg electrically
  Fundamental winding factor kw = 0.9659 (typical: 0.90-0.97 for q=2)


In [4]:
# ─────────────────────────────────────────────────────────────
#  Winding layout table — one sector + full machine
# ─────────────────────────────────────────────────────────────

# Per-sector assignment (6 slots in 45 deg)
slot_phase = ['A', 'A', 'C', 'C', 'B', 'B']
slot_dir   = [+1, +1, -1, -1, +1, +1]   # dir_phA=+1, dir_phC=-1, dir_phB=+1

print('--- ONE 45-deg SECTOR (slots 0-5) ---')
print(f'  {"Slot":>5}  {"Center [deg]":>14}  {"Phase":>6}  {"Direction"}')
print('  ' + '-' * 42)
for si in range(ns):
    theta_c = (si + 0.5) * (theta_s / ns)   # deg
    ph = slot_phase[si]
    d  = slot_dir[si]
    sign = '+' if d > 0 else '-'
    print(f'  {si:>5}  {theta_c:>12.2f} deg  {ph:>6}  {sign}')

print()
print('--- FULL MACHINE — all 48 slots ---')
print(f'  {"Slot":>4}  {"Mech. angle [deg]":>18}  {"Phase+Dir":>10}')
print('  ' + '-' * 40)

for k in range(N_sectors):
    phi_k  = k * theta_s          # sector offset [deg]
    parity = 1 if k % 2 == 0 else -1
    for si in range(ns):
        global_slot = k * ns + si
        theta_c = phi_k + (si + 0.5) * (theta_s / ns)
        ph = slot_phase[si]
        d  = slot_dir[si] * parity
        sign = '+' if d > 0 else '-'
        print(f'  {global_slot:>4}  {theta_c:>16.2f} deg  {ph}{sign:>9}')

print()
print('Pattern explanation:')
print('  Sector 0 (0-45 deg):   A+ A+ C- C- B+ B+   (polarity +1)')
print('  Sector 1 (45-90 deg):  A- A- C+ C+ B- B-   (polarity -1, anti-periodic BC)')
print('  Sector 2 (90-135 deg): A+ A+ C- C- B+ B+   (polarity +1)')
print('  ... alternating every 45 degrees for all 8 sectors.')

--- ONE 45-deg SECTOR (slots 0-5) ---
   Slot    Center [deg]   Phase  Direction
  ------------------------------------------
      0          3.75 deg       A  +
      1         11.25 deg       A  +
      2         18.75 deg       C  -
      3         26.25 deg       C  -
      4         33.75 deg       B  +
      5         41.25 deg       B  +

--- FULL MACHINE — all 48 slots ---
  Slot   Mech. angle [deg]   Phase+Dir
  ----------------------------------------
     0              3.75 deg  A        +
     1             11.25 deg  A        +
     2             18.75 deg  C        -
     3             26.25 deg  C        -
     4             33.75 deg  B        +
     5             41.25 deg  B        +
     6             48.75 deg  A        -
     7             56.25 deg  A        -
     8             63.75 deg  C        +
     9             71.25 deg  C        +
    10             78.75 deg  B        -
    11             86.25 deg  B        -
    12             93.75 deg  A        +


## 3. Electromagnetic Loading

In [5]:
# ─────────────────────────────────────────────────────────────
#  Current density, linear current density, air gap flux density
# ─────────────────────────────────────────────────────────────

# 1) Current density
J_peak = Is / Carea   # [A/m^2]
print(f'Peak current density:  J = Is / Carea = {J_peak/1e6:.3f} MA/m^2')

# 2) Linear current density (electric loading)
D_bore = 2 * R_si * 1e-3       # bore diameter [m]
tau_p  = math.pi * D_bore / (2 * PP)  # pole pitch arc [m]

# Peak of fundamental surface current density [A/m]:
#   A_peak = (3/2) * (4/pi) * q * Is * kw / tau_p
A_peak = (3.0/2.0) * (4.0/math.pi) * q * Is * kw / tau_p
A_rms  = A_peak / math.sqrt(2)

print(f'\nLinear current density (electric loading):')
print(f'  Bore diameter   D_bore = {D_bore*1000:.1f} mm')
print(f'  Pole pitch arc  tau_p  = {tau_p*1000:.2f} mm  = pi*D/(2*PP)')
print(f'  A_peak (fundamental)   = {A_peak/1e3:.2f} kA/m')
print(f'  A_rms                  = {A_rms/1e3:.2f} kA/m')

# 3) Air-gap flux density — analytical, no-load
# Simple reluctance model (Carter factor ignored):
#   B_g = B_r / (1 + mu_r * g / h_m)   [T]
B_g_peak = B_r / (1.0 + mu_r * (g * 1e-3) / (h_m * 1e-3))

# Fundamental of rectangular pulse (width alpha_m, centred at pole axis):
#   B_g1 = (4/pi) * B_g_peak * sin(alpha_m / 2)
alpha_m_elec = mag_frac * math.pi   # electrical half-period = pi
B_g1 = (4.0 / math.pi) * B_g_peak * math.sin(alpha_m_elec / 2.0)

print(f'\nNo-load air gap flux density (analytical, reluctance model):')
print(f'  Remanence         B_r          = {B_r:.4f} T')
print(f'  Under magnet      B_g_peak     = {B_g_peak:.4f} T')
print(f'  Magnet arc        alpha_m      = {math.degrees(alpha_m_elec):.1f} deg (electrical)')
print(f'                                 = {mag_frac*theta_s:.1f} deg (mechanical)')
print(f'  Fundamental       B_g1         = {B_g1:.4f} T')
print(f'  Average (over pole) B_avg      = {mag_frac * B_g_peak:.4f} T')
print(f'\n  Waveform factor (B_g1/B_g_peak) = {B_g1/B_g_peak:.3f}')

Peak current density:  J = Is / Carea = 2.920 MA/m^2

Linear current density (electric loading):
  Bore diameter   D_bore = 148.0 mm
  Pole pitch arc  tau_p  = 58.12 mm  = pi*D/(2*PP)
  A_peak (fundamental)   = 12.70 kA/m
  A_rms                  = 8.98 kA/m

No-load air gap flux density (analytical, reluctance model):
  Remanence         B_r          = 1.1908 T
  Under magnet      B_g_peak     = 0.9980 T
  Magnet arc        alpha_m      = 144.0 deg (electrical)
                                 = 36.0 deg (mechanical)
  Fundamental       B_g1         = 1.2086 T
  Average (over pole) B_avg      = 0.7984 T

  Waveform factor (B_g1/B_g_peak) = 1.211


In [6]:
# ─────────────────────────────────────────────────────────────
#  Air gap B waveform — rectangular + fundamental over 360 deg
# ─────────────────────────────────────────────────────────────

theta_mech = np.linspace(0, 360, 3600)          # mechanical degrees
theta_elec = theta_mech * PP                    # electrical degrees

# Build rectangular waveform (8 poles)
B_rect = np.zeros_like(theta_mech)
alpha_m_mech = mag_frac * theta_s              # magnet arc in mechanical deg = 36 deg

for k in range(N_sectors):
    phi_k  = k * theta_s                       # sector start [deg]
    t_left  = phi_k + theta_ml
    t_right = phi_k + theta_mr
    sign    = +1 if k % 2 == 0 else -1
    mask = (theta_mech >= t_left) & (theta_mech < t_right)
    B_rect[mask] = sign * B_g_peak

# Fundamental sinusoid (electrical angle -> mechanical)
theta_rad = np.radians(theta_mech)
B_fund = B_g1 * np.cos(PP * theta_rad)         # fundamental (cos referenced to 0)

# Correct phase so that fundamental peak aligns with pole centres
# Pole centres at 0, 45, 90, ... deg mech; magnet centred at 22.5 deg each sector
# Shift reference by half-sector (22.5 deg mech = PP*22.5 elec deg)
B_fund_shifted = B_g1 * np.cos(PP * theta_rad - PP * math.radians(theta_s / 2))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(theta_mech, B_rect, color='steelblue', linewidth=1.5,
        label=f'Rectangular waveform (B_g_peak = {B_g_peak:.3f} T)')
ax.plot(theta_mech, B_fund_shifted, color='tomato', linewidth=2, linestyle='--',
        label=f'Fundamental (B_g1 = {B_g1:.3f} T)')
ax.axhline(0, color='k', linewidth=0.5)
ax.fill_between(theta_mech, 0, B_rect,
                where=B_rect > 0, alpha=0.15, color='steelblue')
ax.fill_between(theta_mech, 0, B_rect,
                where=B_rect < 0, alpha=0.15, color='tomato')

# Pole labels
for k in range(N_sectors):
    centre = k * theta_s + theta_s / 2
    label  = 'N' if k % 2 == 0 else 'S'
    ypos   = 0.82 * B_g_peak if k % 2 == 0 else -0.82 * B_g_peak
    ax.text(centre, ypos, label, ha='center', va='center',
            fontsize=11, fontweight='bold',
            color='navy' if label == 'N' else 'darkred')

ax.set_xlabel('Mechanical angle [deg]')
ax.set_ylabel('Air gap flux density B [T]')
ax.set_title('No-load Air Gap Flux Density Waveform (analytical, reluctance model)\n'
             f'8 poles, mag_frac={mag_frac:.2f}, B_r={B_r:.3f} T')
ax.set_xlim(0, 360)
ax.set_xticks(np.arange(0, 361, 45))
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/airgap_B_waveform.png', dpi=150)
plt.show()
print('Saved: results/airgap_B_waveform.png')

Saved: results/airgap_B_waveform.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/4124663175.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Active Material Quantities

In [7]:
# ─────────────────────────────────────────────────────────────
#  Volume and mass estimates from 2D cross-section areas
# ─────────────────────────────────────────────────────────────

theta_s_rad = math.radians(theta_s)   # 45 deg in radians

# --- Stator ---
# Annular sector area (mm^2)
A_stator_annulus = 0.5 * (R_so**2 - R_si**2) * theta_s_rad

# Single slot area: use trapezoidal approximation for winding body
# Angular half-width of slot at R_st: b5 / (2 * R_st) [rad]
slot_half_angle = math.atan(b5 / 2.0 / R_st)  # [rad]
# Annular wedge from R_wb to R_st over ±slot_half_angle
A_slot_geom = 0.5 * (R_st**2 - R_wb**2) * 2 * slot_half_angle   # mm^2

A_stator_iron_sector = A_stator_annulus - ns * A_slot_geom       # mm^2 per sector
A_stator_iron_total  = A_stator_iron_sector * N_sectors * 1e-6   # m^2

# --- Copper in slots ---
fill_factor = Carea / (A_slot_geom * 1e-6)     # dimensionless
A_copper_sector = ns * Carea                   # m^2 per sector
A_copper_total  = A_copper_sector * N_sectors  # m^2

# --- Rotor ---
A_rotor_annulus  = 0.5 * (R_ro**2 - R_ri**2) * theta_s_rad       # mm^2
A_mag_sector     = 0.5 * (R_ro**2 - R_mi**2) * math.radians(theta_mr - theta_ml)  # mm^2
A_rotor_iron_sector = A_rotor_annulus - A_mag_sector              # mm^2 per sector
A_rotor_iron_total  = A_rotor_iron_sector * N_sectors * 1e-6     # m^2

# --- Magnets ---
A_mag_total = A_mag_sector * N_sectors * 1e-6   # m^2

# --- Shaft ---
A_shaft_sector = 0.5 * R_ri**2 * theta_s_rad    # mm^2
A_shaft_total  = A_shaft_sector * N_sectors * 1e-6  # m^2

# --- Volumes [m^3] ---
V_stator_iron = A_stator_iron_total * L_active
V_copper      = A_copper_total      * L_active
V_rotor_iron  = A_rotor_iron_total  * L_active
V_magnet      = A_mag_total         * L_active
V_shaft       = A_shaft_total       * L_active

# --- Masses [kg] ---
m_stator_iron  = V_stator_iron * rho_fe
m_copper       = V_copper      * rho_cu
m_rotor_iron   = V_rotor_iron  * rho_fe
m_magnet       = V_magnet      * rho_pm
m_total_active = m_stator_iron + m_copper + m_rotor_iron + m_magnet

print('Active Material Quantities (2D area x active length, no end-windings)')
print('=' * 62)
print(f'  {"Component":<28} {"Volume [L]":>12}  {"Mass [kg]":>10}')
print('  ' + '-' * 56)
print(f'  {"Stator iron (M350-50A)":<28} {V_stator_iron*1000:>12.3f}  {m_stator_iron:>10.3f}')
print(f'  {"Stator copper (slots)":<28} {V_copper*1000:>12.3f}  {m_copper:>10.3f}')
print(f'  {"Rotor iron":<28} {V_rotor_iron*1000:>12.3f}  {m_rotor_iron:>10.3f}')
print(f'  {"Permanent magnets (NdFeB)":<28} {V_magnet*1000:>12.3f}  {m_magnet:>10.3f}')
print('  ' + '-' * 56)
V_total_active = V_stator_iron + V_copper + V_rotor_iron + V_magnet
print(f'  {"TOTAL (active)":<28} {V_total_active*1000:>12.3f}  {m_total_active:>10.3f}')
print('  ' + '-' * 56)
print(f'  {"Shaft (steel, informative)":<28} {V_shaft*1000:>12.3f}  {"N/A":>10}')
print('=' * 62)
print(f'\n  Conductor area  Carea        = {Carea*1e6:.2f} mm^2')
print(f'  Slot winding area (geom.)   = {A_slot_geom:.2f} mm^2')
print(f'  Slot fill factor            = {fill_factor:.3f}  ({fill_factor*100:.1f}%)')
print(f'\n  Note: Rated power not specified; specific output (kW/kg) TBD.')
print(f'  End-winding mass not included (add ~20-30% to copper mass estimate).')

# Pie chart of mass breakdown
masses  = [m_stator_iron, m_copper, m_rotor_iron, m_magnet]
labels  = ['Stator iron', 'Copper (slots)', 'Rotor iron', 'PM (NdFeB)']
colors  = ['#AAAAAA', '#CC8800', '#888888', '#FF8800']

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    masses, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.75, labeldistance=1.1)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title(f'Active Mass Breakdown — Total = {m_total_active:.2f} kg\n'
             f'(48-slot 8-pole motor, L_active = {L_active*1000:.0f} mm)')
plt.tight_layout()
plt.savefig('results/mass_breakdown.png', dpi=150)
plt.show()
print('Saved: results/mass_breakdown.png')

Active Material Quantities (2D area x active length, no end-windings)
  Component                      Volume [L]   Mass [kg]
  --------------------------------------------------------
  Stator iron (M350-50A)              2.297      17.575
  Stator copper (slots)               0.559       4.975
  Rotor iron                          2.288      17.505
  Permanent magnets (NdFeB)           0.244       1.827
  --------------------------------------------------------
  TOTAL (active)                      5.388      41.881
  --------------------------------------------------------
  Shaft (steel, informative)          0.334         N/A

  Conductor area  Carea        = 68.50 mm^2
  Slot winding area (geom.)   = 152.00 mm^2
  Slot fill factor            = 0.451  (45.1%)

  Note: Rated power not specified; specific output (kW/kg) TBD.
  End-winding mass not included (add ~20-30% to copper mass estimate).
Saved: results/mass_breakdown.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/1902296934.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Machine Cross-Section Drawings

In [8]:
# ─────────────────────────────────────────────────────────────
#  FULL MACHINE CROSS-SECTION  (360 deg, all patches)
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect('equal')

# ---- 1. Shaft ----
shaft_circle = plt.Circle((0, 0), R_ri, color='#666666', zorder=1)
ax.add_patch(shaft_circle)

# ---- 2. Rotor iron (full annulus) ----
rotor_iron = mpatches.Wedge((0, 0), R_ro, 0, 360,
                             width=R_ro - R_ri,
                             color='#AAAAAA', zorder=2)
ax.add_patch(rotor_iron)

# ---- 3. Magnets ----
mag_color_N = '#FF8800'    # orange -> N pole
mag_color_S = '#FFCC88'    # light orange -> S pole
for k in range(N_sectors):
    phi_k  = k * theta_s
    t_left  = phi_k + theta_ml
    t_right = phi_k + theta_mr
    color   = mag_color_N if k % 2 == 0 else mag_color_S
    wedge = mpatches.Wedge((0, 0), R_ro, t_left, t_right,
                            width=h_m, color=color, zorder=3)
    ax.add_patch(wedge)
    # N/S label on magnets
    angle_mid = math.radians(phi_k + theta_s / 2)
    r_label   = (R_ro - h_m / 2)
    label_txt = 'N' if k % 2 == 0 else 'S'
    ax.text(r_label * math.cos(angle_mid),
            r_label * math.sin(angle_mid),
            label_txt, ha='center', va='center',
            fontsize=9, fontweight='bold', color='black', zorder=10)

# ---- 4. Air gap (very light blue) ----
air_gap_patch = mpatches.Wedge((0, 0), R_si, 0, 360,
                                width=R_si - R_ro,
                                color='#F0F8FF', zorder=4)
ax.add_patch(air_gap_patch)

# ---- 5. Stator iron (full annulus, grey) ----
stator_iron = mpatches.Wedge((0, 0), R_so, 0, 360,
                              width=R_so - R_si,
                              color='#CCCCCC', zorder=5)
ax.add_patch(stator_iron)

# ---- 6. Stator slots ----
slot_phase_list = ['A', 'A', 'C', 'C', 'B', 'B']
slot_dir_list   = [+1, +1, -1, -1, +1, +1]
colors_pos = {'A': '#FF3333', 'B': '#3355EE', 'C': '#22AA22'}
colors_neg = {'A': '#FFBBBB', 'B': '#AABBFF', 'C': '#AADDAA'}

angular_step  = theta_s / ns          # 7.5 deg per slot
slot_half_deg = angular_step * 0.72 / 2   # draw 72% of pitch, leave 28% for iron

for k in range(N_sectors):
    phi_k  = k * theta_s
    parity = +1 if k % 2 == 0 else -1
    for si in range(ns):
        theta_c = phi_k + (si + 0.5) * angular_step   # slot centre [deg]
        ph  = slot_phase_list[si]
        d   = slot_dir_list[si] * parity
        color = colors_pos[ph] if d > 0 else colors_neg[ph]
        wedge = mpatches.Wedge(
            (0, 0), R_st,
            theta_c - slot_half_deg, theta_c + slot_half_deg,
            width=R_st - R_wb,
            color=color, zorder=6)
        ax.add_patch(wedge)

# ---- 7. Boundary circles ----
for r, lw, ls in [(R_so, 1.2, '-'), (R_si, 0.7, '-'),
                   (R_ro, 0.5, '--'), (R_sb, 0.4, ':'),
                   (R_ri, 0.7, '-')]:
    circ = plt.Circle((0, 0), r, color='black', fill=False,
                       linewidth=lw, linestyle=ls, zorder=11)
    ax.add_patch(circ)

# ---- 8. Dimension annotations ----
# R_so arrow (along 0 deg axis)
ax.annotate('', xy=(R_so, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='darkblue', lw=1.5), zorder=12)
ax.text(R_so / 2, 4, f'R_so = {R_so:.0f} mm', color='darkblue',
        fontsize=8, ha='center', zorder=12)

# Air gap annotation (along 90 deg axis)
ax.annotate('', xy=(0, R_si), xytext=(0, R_ro),
            arrowprops=dict(arrowstyle='<->', color='green', lw=1.5), zorder=12)
ax.text(6, (R_si + R_ro) / 2, f'g={g} mm', color='green',
        fontsize=7, va='center', zorder=12)

# R_ri arrow (along 180 deg)
ax.annotate('', xy=(-R_ri, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#444444', lw=1.2), zorder=12)
ax.text(-R_ri / 2, -5, f'R_ri={R_ri:.0f}', color='#444444',
        fontsize=7, ha='center', zorder=12)

# ---- 9. Legend ----
legend_elements = [
    mpatches.Patch(facecolor=colors_pos['A'], label='Phase A (+)'),
    mpatches.Patch(facecolor=colors_neg['A'], label='Phase A (-)'),
    mpatches.Patch(facecolor=colors_pos['B'], label='Phase B (+)'),
    mpatches.Patch(facecolor=colors_neg['B'], label='Phase B (-)'),
    mpatches.Patch(facecolor=colors_pos['C'], label='Phase C (+)'),
    mpatches.Patch(facecolor=colors_neg['C'], label='Phase C (-)'),
    mpatches.Patch(facecolor=mag_color_N,     label='PM — N pole'),
    mpatches.Patch(facecolor=mag_color_S,     label='PM — S pole'),
    mpatches.Patch(facecolor='#CCCCCC',       label='Stator iron'),
    mpatches.Patch(facecolor='#AAAAAA',       label='Rotor iron'),
    mpatches.Patch(facecolor='#666666',       label='Shaft'),
]
ax.legend(handles=legend_elements, loc='upper right',
          fontsize=7.5, ncol=2,
          bbox_to_anchor=(1.0, 1.0),
          framealpha=0.9)

ax.set_xlim(-125, 125)
ax.set_ylim(-125, 125)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    'TractionMachinePM2D — Full Machine Cross-Section\n'
    f'48 slots, 8 poles (4 pole pairs), OD = {2*R_so:.0f} mm, '
    f'ID = {2*R_ri:.0f} mm, g = {g} mm',
    fontsize=12, fontweight='bold')

# Compass rose (top-centre)
ax.text(0, 118, 'N', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(118, 0, 'E', ha='center', va='center', fontsize=10)
ax.text(0, -118, 'S', ha='center', va='center', fontsize=10)
ax.text(-118, 0, 'W', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('results/cross_section_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/cross_section_full.png')

Saved: results/cross_section_full.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/2260549932.py:138: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ─────────────────────────────────────────────────────────────
#  DETAIL VIEW — stator slot & one rotor pole
# ─────────────────────────────────────────────────────────────

fig, (ax_slot, ax_rotor) = plt.subplots(1, 2, figsize=(14, 7))

# ============================================================
#  LEFT — Stator slot detail (slots 2 & 3, Phase C)
# ============================================================
ax = ax_slot
ax.set_aspect('equal')

# Draw 3 slots (slots 1, 2, 3) to give context; highlight 2 & 3
# Show the stator iron background for slots 1-3
stator_bg = mpatches.Wedge(
    (0, 0), R_so,
    1 * angular_step, 4 * angular_step,   # 7.5 to 30 deg
    width=R_so - R_si, color='#CCCCCC', zorder=1)
ax.add_patch(stator_bg)

# Draw the 3 highlighted slots
slot_colors_detail = [
    (colors_neg['A'], 'A−'),   # slot 1: A-
    (colors_neg['C'], 'C−'),   # slot 2: C- (sector 0)
    (colors_neg['C'], 'C−'),   # slot 3: C-
]
for i, (si, (col, lbl)) in enumerate(zip([1, 2, 3], slot_colors_detail)):
    theta_c = (si + 0.5) * angular_step
    wedge = mpatches.Wedge(
        (0, 0), R_st,
        theta_c - slot_half_deg, theta_c + slot_half_deg,
        width=R_st - R_wb,
        color=col, zorder=3, alpha=0.9)
    ax.add_patch(wedge)
    # Slot label
    angle_c = math.radians(theta_c)
    r_mid   = (R_wb + R_st) / 2
    ax.text(r_mid * math.cos(angle_c), r_mid * math.sin(angle_c),
            lbl, ha='center', va='center', fontsize=9,
            fontweight='bold', color='black', zorder=5)

# Slot opening (narrow wedge at bore)
for si in [1, 2, 3]:
    theta_c   = (si + 0.5) * angular_step
    b1_angle  = math.degrees(math.atan(b1 / 2.0 / R_si))
    opening   = mpatches.Wedge(
        (0, 0), R_wb,
        theta_c - b1_angle, theta_c + b1_angle,
        width=h1 + h2, color='white', zorder=4)
    ax.add_patch(opening)

# Bore circle
bore_arc = plt.Circle((0, 0), R_si, color='k', fill=False, lw=0.8, zorder=6)
ax.add_patch(bore_arc)
outer_arc = plt.Circle((0, 0), R_so, color='k', fill=False, lw=1.0, zorder=6)
ax.add_patch(outer_arc)
wb_arc = plt.Circle((0, 0), R_wb, color='k', fill=False, lw=0.5,
                     linestyle='--', zorder=6)
ax.add_patch(wb_arc)
st_arc = plt.Circle((0, 0), R_st, color='k', fill=False, lw=0.5,
                     linestyle='--', zorder=6)
ax.add_patch(st_arc)

# Dimension lines for slot 2
si2   = 2
tc2   = (si2 + 0.5) * angular_step   # 18.75 deg
tc2r  = math.radians(tc2)

# h5 dimension (radial, right side of slot 2)
r_dim  = R_wb + 1.5
angle_r = math.radians(tc2 + slot_half_deg + 1.5)
ax.annotate('', xy=(R_st * math.cos(tc2r), R_st * math.sin(tc2r)),
            xytext=(R_wb * math.cos(tc2r), R_wb * math.sin(tc2r)),
            arrowprops=dict(arrowstyle='<->', color='darkblue', lw=1.2), zorder=7)
ax.text((R_wb + h5 / 2 * 0.5) * math.cos(tc2r - 0.15),
        (R_wb + h5 / 2 * 0.5) * math.sin(tc2r - 0.15),
        f'h5={h5:.0f}mm', fontsize=7.5, color='darkblue', zorder=8,
        ha='right')

# b5 dimension (tangential at slot top)
tc3r  = math.radians((3 + 0.5) * angular_step)
ax.annotate('', xy=(R_st * math.cos(tc2r + 0.04), R_st * math.sin(tc2r + 0.04)),
            xytext=(R_st * math.cos(tc2r - 0.04), R_st * math.sin(tc2r - 0.04)),
            arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=1.2), zorder=7)
ax.text(R_st * math.cos(tc2r) + 2,
        R_st * math.sin(tc2r) + 2,
        f'b5={b5}mm', fontsize=7.5, color='darkgreen', zorder=8)

# b1 label (slot opening)
ax.text(R_si * math.cos(tc2r) - 2, R_si * math.sin(tc2r) - 3,
        f'b1={b1}mm', fontsize=7.5, color='purple', zorder=8, ha='center')

# b4 label (winding bottom width)
ax.text(R_wb * math.cos(tc2r) + 3, R_wb * math.sin(tc2r),
        f'b4={b4}mm', fontsize=7.5, color='brown', zorder=8)

ax.set_xlim(60, 105)
ax.set_ylim(0, 50)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
ax.set_title('Stator Slot Detail\n(slots 1-3, centred on Phase C, sector 0)',
             fontweight='bold')
ax.grid(True, alpha=0.2)

# Add slot opening / closing heights annotation
info_txt = (f'Slot geometry:\n'
            f'  b1 = {b1} mm (opening)\n'
            f'  b4 = {b4} mm (bottom)\n'
            f'  b5 = {b5} mm (top)\n'
            f'  h5 = {h5} mm (height)\n'
            f'  h1 = {h1} mm, h2 = {h2} mm\n'
            f'  Fill factor = {fill_factor:.2f}')
ax.text(0.02, 0.97, info_txt, transform=ax.transAxes,
        fontsize=7.5, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# ============================================================
#  RIGHT — One rotor pole (45-degree sector)
# ============================================================
ax = ax_rotor
ax.set_aspect('equal')

# Sector 0: 0 to 45 degrees
# Rotor iron
rotor_sector = mpatches.Wedge(
    (0, 0), R_ro, 0, 45,
    width=R_ro - R_ri, color='#AAAAAA', zorder=1)
ax.add_patch(rotor_sector)

# Shaft
shaft_s = mpatches.Wedge(
    (0, 0), R_ri, -5, 50,
    color='#666666', zorder=0)
ax.add_patch(shaft_s)

# Magnet (N pole, sector 0)
mag_sector = mpatches.Wedge(
    (0, 0), R_ro, theta_ml, theta_mr,
    width=h_m, color=mag_color_N, zorder=2)
ax.add_patch(mag_sector)

# Air gap
ag_sector = mpatches.Wedge(
    (0, 0), R_si, -2, 47,
    width=R_si - R_ro, color='#F0F8FF', zorder=3)
ax.add_patch(ag_sector)

# Boundary circles
for r, lw, ls, lbl, r_off, a_off in [
    (R_ro, 1.0, '-',  f'R_ro={R_ro}mm', 2,  22.5),
    (R_mi, 0.8, '--', f'R_mi={R_mi}mm', -5, 22.5),
    (R_ri, 0.8, '-',  f'R_ri={R_ri}mm', -5, 10),
    (R_si, 0.6, ':',  f'R_si={R_si}mm', 2,  22.5),
]:
    circ = plt.Circle((0, 0), r, color='k', fill=False,
                       lw=lw, linestyle=ls, zorder=5)
    ax.add_patch(circ)
    a_r = math.radians(a_off)
    ax.text((r + r_off) * math.cos(a_r), (r + r_off) * math.sin(a_r),
            lbl, fontsize=7, color='navy', ha='center', zorder=6)

# Sector boundary lines
for angle_deg in [0, 45]:
    ar = math.radians(angle_deg)
    ax.plot([0, R_so * math.cos(ar)], [0, R_so * math.sin(ar)],
            'k--', lw=0.7, alpha=0.5, zorder=4)

# Magnet edge lines
for angle_deg in [theta_ml, theta_mr]:
    ar = math.radians(angle_deg)
    ax.plot([R_mi * math.cos(ar), R_ro * math.cos(ar)],
            [R_mi * math.sin(ar), R_ro * math.sin(ar)],
            'k-', lw=1.0, zorder=6)

# h_m dimension arrow (radial at mid-angle)
am = math.radians((theta_ml + theta_mr) / 2)
ax.annotate('', xy=(R_ro * math.cos(am), R_ro * math.sin(am)),
            xytext=(R_mi * math.cos(am), R_mi * math.sin(am)),
            arrowprops=dict(arrowstyle='<->', color='red', lw=1.5), zorder=7)
ax.text((R_mi + h_m / 2) * math.cos(am) + 2,
        (R_mi + h_m / 2) * math.sin(am),
        f'h_m={h_m}mm', fontsize=8, color='red', zorder=8)

# Arc span annotation
arc_angle_text = (theta_mr - theta_ml)
ax.text(R_ro * math.cos(am) + 4, R_ro * math.sin(am) + 4,
        f'alpha_m = {arc_angle_text:.0f} deg\n= {mag_frac*100:.0f}% of pole pitch',
        fontsize=8, color='darkorange', zorder=8,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

# N label
ax.text((R_mi + h_m * 1.5) * math.cos(am),
        (R_mi + h_m * 1.5) * math.sin(am),
        'N', ha='center', va='center',
        fontsize=14, fontweight='bold', color='navy', zorder=9)

ax.set_xlim(-5, 80)
ax.set_ylim(-5, 80)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
ax.set_title('Rotor Pole Detail — One 45-deg Sector\n'
             f'Inset PM (NdFeB), B_r={B_r:.3f} T, H_PM={H_PM/1e3:.0f} kA/m',
             fontweight='bold')
ax.grid(True, alpha=0.2)

info_rotor = (f'Magnet parameters:\n'
              f'  h_m    = {h_m} mm\n'
              f'  R_mi   = {R_mi} mm\n'
              f'  R_ro   = {R_ro} mm\n'
              f'  span   = {theta_mr-theta_ml} deg ({mag_frac*100:.0f}% pitch)\n'
              f'  B_r    = {B_r:.4f} T\n'
              f'  mu_r   = {mu_r}\n'
              f'  B_g    = {B_g_peak:.4f} T')
ax.text(0.02, 0.97, info_rotor, transform=ax.transAxes,
        fontsize=8, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('results/detail_slot_rotor.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/detail_slot_rotor.png')

Saved: results/detail_slot_rotor.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/837448041.py:220: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. FEM Simulation Results

Results loaded from the transient 2D FEM simulation (ElmerFEM).  
The simulation models one 45-deg sector with anti-periodic boundary conditions,  
driven at 1500 rpm (25 Hz mechanical, 100 Hz electrical).  
All quantities are scaled to the full machine: `SCALE = 8 sectors × 170 mm = 1.36 m`.

In [10]:
# ─────────────────────────────────────────────────────────────
#  Load and plot torque waveform from scalars.dat
# ─────────────────────────────────────────────────────────────

from pathlib import Path

dat = np.loadtxt('results/scalars.dat')

# Column mapping (1-indexed in names file, 0-indexed here):
#   0: res: eddy current power
#   1: res: electromagnetic field energy
#   2: res: fourier loss total
#   3: res: fourier loss 1
#   4: res: fourier loss 2
#   5: res: air gap torque
#   6: res: inertial volume
#   7: res: inertial moment
#   8: res: group 1 torque

n_steps = dat.shape[0]
print(f'Loaded scalars.dat: {n_steps} timesteps, {dat.shape[1]} columns')

# Time axis
t    = np.arange(1, n_steps + 1) * dt    # seconds
t_ms = t * 1000                           # milliseconds

# Torque from group 1 (rotor body): raw = N·m/m per sector (2D Elmer output)
torque_raw       = dat[:, 8]               # N·m/m, per sector
torque_full      = torque_raw * SCALE      # N·m, full machine

# Air gap torque
airgap_torque_raw  = dat[:, 5]             # N·m/m, per sector
airgap_torque_full = airgap_torque_raw * SCALE

# Mechanical period
T_mech_ms = 1000.0 / fme   # 40 ms

# ---- Plot ----
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(t_ms, torque_full, 'b-', linewidth=1.6, label='Group 1 torque (rotor body)')
ax.plot(t_ms, airgap_torque_full, 'r--', linewidth=1.2, alpha=0.75,
        label='Air gap torque')

# Cycle boundary
ax.axvline(x=T_mech_ms, color='gray', linestyle=':', alpha=0.6, label='End of cycle 1')

# Mean over cycle 2
cycle2_mask = t > 1.0 / fme
T_mean    = np.mean(torque_full[cycle2_mask])
T_max     = np.max(torque_full[cycle2_mask])
T_min     = np.min(torque_full[cycle2_mask])
T_ripple  = (T_max - T_min) / T_mean * 100   # %
P_mech    = T_mean * (2 * math.pi * rpm / 60) / 1000  # kW

ax.axhline(y=T_mean, color='green', linestyle='--', alpha=0.6, linewidth=1.2,
           label=f'Mean torque = {T_mean:.1f} N·m')

# Annotation box
annot = (f'Cycle 2 statistics:\n'
         f'  Mean torque = {T_mean:.1f} N·m\n'
         f'  Max         = {T_max:.1f} N·m\n'
         f'  Min         = {T_min:.1f} N·m\n'
         f'  Ripple      = {T_ripple:.1f}%\n'
         f'  Mech. power = {P_mech:.2f} kW')
ax.text(0.98, 0.05, annot, transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))

ax.set_xlabel('Time [ms]')
ax.set_ylabel('Torque [N·m]')
ax.set_title(
    'Electromagnetic Torque — Full Machine (8 sectors × 170 mm)\n'
    f'n = {rpm} rpm, Is = {Is:.0f} A (placeholder), SCALE = {SCALE:.2f} m',
    fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, t_ms[-1])

plt.tight_layout()
plt.savefig('results/torque_waveform.png', dpi=150)
plt.show()
print('Saved: results/torque_waveform.png')
print()
print(f'Mean torque (cycle 2):  T_mean = {T_mean:.2f} N·m')
print(f'Torque ripple:          {T_ripple:.1f}%')
print(f'Mechanical power:       P = T*omega = {P_mech:.3f} kW')
print(f'  (at n = {rpm} rpm, omega = {2*math.pi*rpm/60:.2f} rad/s)')

Loaded scalars.dat: 241 timesteps, 9 columns
Saved: results/torque_waveform.png

Mean torque (cycle 2):  T_mean = 112.22 N·m
Torque ripple:          24.9%
Mechanical power:       P = T*omega = 17.627 kW
  (at n = 1500 rpm, omega = 157.08 rad/s)


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/4038209666.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ─────────────────────────────────────────────────────────────
#  Iron losses from loss-2d.dat
# ─────────────────────────────────────────────────────────────

loss_raw = np.loadtxt('results/loss-2d.dat', comments='!')

# Parse: body_id, loss_hyst [W/m per sector], loss_eddy [W/m per sector]
loss_dict = {int(r[0]): (r[1], r[2]) for r in loss_raw}

print('Body IDs found in loss-2d.dat:', list(loss_dict.keys()))
print('  Body  1 = Stator lamination')
print('  Body 10 = Rotor lamination')
print()

# Stator losses
if 1 in loss_dict:
    s_h_raw, s_e_raw = loss_dict[1]           # W/m per sector
    S_hyst_W  = s_h_raw * SCALE               # W full machine
    S_eddy_W  = s_e_raw * SCALE
    S_total_W = S_hyst_W + S_eddy_W
else:
    S_hyst_W = S_eddy_W = S_total_W = 0.0
    print('WARNING: body 1 (stator) not found in loss-2d.dat')

# Rotor losses
if 10 in loss_dict:
    r_h_raw, r_e_raw = loss_dict[10]
    R_hyst_W  = r_h_raw * SCALE
    R_eddy_W  = r_e_raw * SCALE
    R_total_W = R_hyst_W + R_eddy_W
else:
    R_hyst_W = R_eddy_W = R_total_W = 0.0
    print('WARNING: body 10 (rotor) not found in loss-2d.dat')

total_iron_W = S_total_W + R_total_W

print('Iron Losses — Full Machine (scaled from 45-deg sector FEM)')
print('=' * 68)
print(f'  {"Component":<25} {"Hysteresis [W]":>16} {"Eddy [W]":>12} {"Total [W]":>12}')
print('  ' + '-' * 64)
print(f'  {"Stator iron":<25} {S_hyst_W:>16.2f} {S_eddy_W:>12.2f} {S_total_W:>12.2f}')
print(f'  {"Rotor iron":<25} {R_hyst_W:>16.2f} {R_eddy_W:>12.2f} {R_total_W:>12.2f}')
print('  ' + '-' * 64)
print(f'  {"TOTAL":<25} {S_hyst_W+R_hyst_W:>16.2f}'
      f' {S_eddy_W+R_eddy_W:>12.2f} {total_iron_W:>12.2f}')
print('=' * 68)
print()
print(f'  Scaling: SCALE = {N_sectors} sectors x {L_active*1000:.0f} mm = {SCALE:.3f} m')
print(f'  Is = {Is} A (placeholder); iron losses scale approximately as B^2 ~ Is^2')
if S_total_W > 0 and P_mech > 0:
    eff_approx = P_mech / (P_mech + total_iron_W / 1000) * 100
    print(f'\n  Indicative efficiency (ignoring copper loss, friction, windage):')
    print(f'    P_mech = {P_mech:.2f} kW')
    print(f'    P_iron = {total_iron_W/1000:.3f} kW')
    print(f'    eta >= {eff_approx:.2f}% (iron loss only)')

Body IDs found in loss-2d.dat: [1, 10]
  Body  1 = Stator lamination
  Body 10 = Rotor lamination

Iron Losses — Full Machine (scaled from 45-deg sector FEM)
  Component                   Hysteresis [W]     Eddy [W]    Total [W]
  ----------------------------------------------------------------
  Stator iron                         130.42       126.78       257.20
  Rotor iron                            0.08         0.59         0.67
  ----------------------------------------------------------------
  TOTAL                               130.50       127.37       257.87

  Scaling: SCALE = 8 sectors x 170 mm = 1.360 m
  Is = 200.0 A (placeholder); iron losses scale approximately as B^2 ~ Is^2

  Indicative efficiency (ignoring copper loss, friction, windage):
    P_mech = 17.63 kW
    P_iron = 0.258 kW
    eta >= 98.56% (iron loss only)


In [12]:
# ─────────────────────────────────────────────────────────────
#  Iron loss breakdown — bar chart
# ─────────────────────────────────────────────────────────────

categories = ['Stator\nHysteresis', 'Stator\nEddy',
              'Rotor\nHysteresis', 'Rotor\nEddy']
values     = [S_hyst_W, S_eddy_W, R_hyst_W, R_eddy_W]
bar_colors = ['#CC4444', '#FF8888', '#4466CC', '#88AAFF']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- Left: individual bar chart ----
ax = axes[0]
bars = ax.bar(categories, values, color=bar_colors, edgecolor='black',
              linewidth=0.7, width=0.6)
# Value labels on bars
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            f'{val:.2f} W', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Power loss [W]')
ax.set_title('Iron Loss Components\n(full machine, placeholder Is=200A)')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(values) * 1.25)

# ---- Right: stacked bar (stator vs rotor) ----
ax2 = axes[1]
bodies    = ['Stator', 'Rotor', 'Total']
hyst_vals = [S_hyst_W, R_hyst_W, S_hyst_W + R_hyst_W]
eddy_vals = [S_eddy_W, R_eddy_W, S_eddy_W + R_eddy_W]

x = np.arange(len(bodies))
w = 0.45
b1_bars = ax2.bar(x, hyst_vals, w, label='Hysteresis loss',
                   color=['#CC4444', '#4466CC', '#886600'], edgecolor='black', lw=0.7)
b2_bars = ax2.bar(x, eddy_vals, w, bottom=hyst_vals, label='Eddy current loss',
                   color=['#FF8888', '#88AAFF', '#FFCC44'], edgecolor='black', lw=0.7)

# Total labels
totals = [S_total_W, R_total_W, total_iron_W]
for xi, tot in zip(x, totals):
    ax2.text(xi, tot + max(totals) * 0.01,
             f'{tot:.2f} W', ha='center', va='bottom', fontsize=9)

ax2.set_xticks(x)
ax2.set_xticklabels(bodies)
ax2.set_ylabel('Power loss [W]')
ax2.set_title('Iron Loss: Stator vs Rotor (stacked)\n(full machine, placeholder Is=200A)')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, max(totals) * 1.25)

plt.suptitle(
    f'Iron Loss Breakdown — Full Machine (placeholder Is={Is:.0f} A)\n'
    f'n={rpm} rpm, f_el={fel:.0f} Hz, M350-50A lamination steel',
    fontsize=11, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('results/iron_loss_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/iron_loss_breakdown.png')

Saved: results/iron_loss_breakdown.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/87609247.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ─────────────────────────────────────────────────────────────
#  Additional: Field energy & eddy power transient
# ─────────────────────────────────────────────────────────────

eddy_power   = dat[:, 0]   # W/m per sector
field_energy = dat[:, 1]   # J/m per sector

eddy_power_full   = eddy_power   * SCALE  # W full machine
field_energy_full = field_energy * SCALE  # J full machine

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t_ms, eddy_power_full, 'b-', linewidth=1.4, label='Eddy current power')
ax1.set_ylabel('Eddy power [W]')
ax1.set_title('Eddy Current Power & Field Energy Transient\n'
              f'(full machine, SCALE={SCALE:.2f} m)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=T_mech_ms, color='gray', linestyle=':', alpha=0.5)

ax2.plot(t_ms, field_energy_full, 'r-', linewidth=1.4, label='EM field energy')
ax2.set_xlabel('Time [ms]')
ax2.set_ylabel('Field energy [J]')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.axvline(x=T_mech_ms, color='gray', linestyle=':', alpha=0.5)

# Cycle-2 means
eddy_mean   = np.mean(eddy_power_full[cycle2_mask])
energy_mean = np.mean(field_energy_full[cycle2_mask])
ax1.axhline(y=eddy_mean, color='navy', linestyle='--', alpha=0.5,
            label=f'Mean (cycle 2) = {eddy_mean:.1f} W')
ax2.axhline(y=energy_mean, color='darkred', linestyle='--', alpha=0.5,
            label=f'Mean (cycle 2) = {energy_mean:.3f} J')
ax1.legend(fontsize=8)
ax2.legend(fontsize=8)

print(f'Cycle-2 mean eddy current power  = {eddy_mean:.2f} W')
print(f'Cycle-2 mean field energy         = {energy_mean:.4f} J')

plt.tight_layout()
plt.savefig('results/eddy_field_transient.png', dpi=150)
plt.show()
print('Saved: results/eddy_field_transient.png')

Cycle-2 mean eddy current power  = 15.62 W
Cycle-2 mean field energy         = 90.5812 J


Saved: results/eddy_field_transient.png


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/1463965140.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary

Key results from the 2D FEM transient simulation and analytical calculations.

In [14]:
# ─────────────────────────────────────────────────────────────
#  SUMMARY TABLE
# ─────────────────────────────────────────────────────────────

print('=' * 70)
print('  TractionMachinePM2D — Engineering Summary Report')
print('  48-slot, 8-pole, 3-phase Inset PM Traction Motor')
print('=' * 70)

print('\n  TOPOLOGY')
print(f'  {"Slots / Poles / Phases":<40} {Qs} / {Qp} / {m}')
print(f'  {"Slots per pole per phase  q":<40} {q}')
print(f'  {"FEM sector angle":<40} {theta_s:.0f} deg (1/8 machine)')

print('\n  GEOMETRY')
print(f'  {"Stator OD / ID":<40} {2*R_so:.0f} mm / {2*R_si:.0f} mm')
print(f'  {"Rotor OD / Shaft OD":<40} {2*R_ro:.2f} mm / {2*R_ri:.0f} mm')
print(f'  {"Air gap  g":<40} {g:.2f} mm')
print(f'  {"Active length  L":<40} {L_active*1000:.0f} mm')
print(f'  {"Magnet radial depth  h_m":<40} {h_m:.1f} mm')
print(f'  {"Magnet arc fraction":<40} {mag_frac:.0%}  ({mag_frac*theta_s:.1f} deg mechanical)')

print('\n  ELECTRICAL')
print(f'  {"Speed":<40} {rpm} rpm')
print(f'  {"Electrical frequency":<40} {fel:.0f} Hz')
print(f'  {"Winding factor  kw":<40} {kw:.4f}')
print(f'  {"Peak current  Is (placeholder)":<40} {Is:.0f} A')
print(f'  {"Peak current density  J":<40} {J_peak/1e6:.2f} MA/m^2')
print(f'  {"Linear current density  A_peak":<40} {A_peak/1e3:.2f} kA/m')

print('\n  MAGNET (NdFeB)')
print(f'  {"Remanence  B_r":<40} {B_r:.4f} T')
print(f'  {"Coercivity  H_PM":<40} {H_PM/1e3:.0f} kA/m')
print(f'  {"Air gap flux density  B_g (peak)":<40} {B_g_peak:.4f} T')
print(f'  {"Fundamental  B_g1":<40} {B_g1:.4f} T')

print('\n  ACTIVE MATERIAL MASS')
print(f'  {"Stator iron (M350-50A)":<40} {m_stator_iron:.3f} kg')
print(f'  {"Copper in slots":<40} {m_copper:.3f} kg')
print(f'  {"Rotor iron":<40} {m_rotor_iron:.3f} kg')
print(f'  {"Permanent magnets":<40} {m_magnet:.3f} kg')
print(f'  {"Total active mass":<40} {m_total_active:.3f} kg')
print(f'  {"Slot fill factor":<40} {fill_factor:.3f} ({fill_factor*100:.1f}%)')

print('\n  FEM RESULTS (transient, cycle 2 steady state)')
print(f'  {"Mean electromagnetic torque":<40} {T_mean:.2f} N·m')
print(f'  {"Torque ripple":<40} {T_ripple:.2f}%')
print(f'  {"Mechanical shaft power":<40} {P_mech:.3f} kW')
print(f'  {"Stator iron losses (total)":<40} {S_total_W:.2f} W')
print(f'    {"  -- hysteresis":<38} {S_hyst_W:.2f} W')
print(f'    {"  -- eddy current":<38} {S_eddy_W:.2f} W')
print(f'  {"Rotor iron losses (total)":<40} {R_total_W:.4f} W')
print(f'  {"Total iron losses":<40} {total_iron_W:.2f} W')
print(f'  {"Mean eddy current power":<40} {eddy_mean:.2f} W')
print(f'  {"Mean EM field energy":<40} {energy_mean:.4f} J')
print(f'\n  {"Scaling factor  SCALE":<40} {SCALE:.3f} m  ({N_sectors} x {L_active*1000:.0f}mm)')

print('\n  NOTES')
print('  * Is=200A is a placeholder; actual rated current TBD.')
print('  * Iron losses use M350-50A loss data from .pmf file.')
print('  * Rotor iron loss is very low: rotor sees predominantly DC flux.')
print('  * End-winding mass, bearing losses, windage not included.')
print('  * Copper losses not simulated (AC winding resistance TBD).')
print('=' * 70)

  TractionMachinePM2D — Engineering Summary Report
  48-slot, 8-pole, 3-phase Inset PM Traction Motor

  TOPOLOGY
  Slots / Poles / Phases                   48 / 8 / 3
  Slots per pole per phase  q              2
  FEM sector angle                         45 deg (1/8 machine)

  GEOMETRY
  Stator OD / ID                           220 mm / 148 mm
  Rotor OD / Shaft OD                      146.50 mm / 50 mm
  Air gap  g                               0.75 mm
  Active length  L                         170 mm
  Magnet radial depth  h_m                 4.0 mm
  Magnet arc fraction                      80%  (36.0 deg mechanical)

  ELECTRICAL
  Speed                                    1500 rpm
  Electrical frequency                     100 Hz
  Winding factor  kw                       0.9659
  Peak current  Is (placeholder)           200 A
  Peak current density  J                  2.92 MA/m^2
  Linear current density  A_peak           12.70 kA/m

  MAGNET (NdFeB)
  Remanence  B_r            

In [15]:
# ─────────────────────────────────────────────────────────────
#  Summary dashboard figure
# ─────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(14, 9))
fig.suptitle('TractionMachinePM2D — Engineering Dashboard\n'
             '48-slot 8-pole Inset PM Motor, n=1500 rpm, L=170 mm',
             fontsize=13, fontweight='bold', y=0.98)

# Grid layout
gs = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.35)

# ---- (0,0): Torque waveform ----
ax0 = fig.add_subplot(gs[0, 0:2])
ax0.plot(t_ms, torque_full, 'b-', lw=1.4, label='Group 1 torque')
ax0.plot(t_ms, airgap_torque_full, 'r--', lw=1.0, alpha=0.7, label='Air gap torque')
ax0.axhline(y=T_mean, color='green', linestyle='--', alpha=0.6, lw=1.2)
ax0.set_xlabel('Time [ms]')
ax0.set_ylabel('Torque [N·m]')
ax0.set_title(f'Torque  |  Mean={T_mean:.1f} N·m, Ripple={T_ripple:.1f}%,'
              f' P={P_mech:.2f} kW')
ax0.legend(fontsize=8)
ax0.grid(True, alpha=0.3)
ax0.set_xlim(0, t_ms[-1])

# ---- (0,2): Air gap B waveform (1 pole pair shown) ----
ax1 = fig.add_subplot(gs[0, 2])
theta_detail = np.linspace(0, 90, 900)   # 2 poles = 90 deg mech
B_rect_d = np.zeros_like(theta_detail)
for k in range(2):   # 2 poles in this window
    phi_k  = k * theta_s
    t_l    = phi_k + theta_ml
    t_r    = phi_k + theta_mr
    sign   = +1 if k % 2 == 0 else -1
    mask   = (theta_detail >= t_l) & (theta_detail < t_r)
    B_rect_d[mask] = sign * B_g_peak

theta_d_rad  = np.radians(theta_detail)
B_fund_d     = B_g1 * np.cos(PP * theta_d_rad - PP * math.radians(theta_s / 2))

ax1.plot(theta_detail, B_rect_d, 'steelblue', lw=1.5, label=f'B_g={B_g_peak:.3f}T')
ax1.plot(theta_detail, B_fund_d, 'tomato', lw=1.5, ls='--',
         label=f'B_g1={B_g1:.3f}T')
ax1.axhline(0, color='k', lw=0.5)
ax1.fill_between(theta_detail, 0, B_rect_d, alpha=0.1, color='steelblue')
ax1.set_xlabel('Mech. angle [deg]')
ax1.set_ylabel('B [T]')
ax1.set_title(f'Air Gap Flux (2 poles)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 90)

# ---- (1,0): Mass breakdown pie ----
ax2 = fig.add_subplot(gs[1, 0])
masses_pie  = [m_stator_iron, m_copper, m_rotor_iron, m_magnet]
labels_pie  = ['St. iron', 'Copper', 'Rot. iron', 'PM']
colors_pie  = ['#AAAAAA', '#CC8800', '#888888', '#FF8800']
wedges2, texts2, at2 = ax2.pie(
    masses_pie, labels=labels_pie, colors=colors_pie,
    autopct='%1.0f%%', startangle=90,
    pctdistance=0.70, textprops={'fontsize': 8})
ax2.set_title(f'Active Mass\n(total {m_total_active:.2f} kg)')

# ---- (1,1): Iron loss bars ----
ax3 = fig.add_subplot(gs[1, 1])
cats2 = ['St. Hyst.', 'St. Eddy', 'Rot. Hyst.', 'Rot. Eddy']
vals2 = [S_hyst_W, S_eddy_W, R_hyst_W, R_eddy_W]
bcols = ['#CC4444', '#FF9999', '#4466CC', '#99AAFF']
bars2 = ax3.bar(cats2, vals2, color=bcols, edgecolor='k', lw=0.6)
for b, v in zip(bars2, vals2):
    ax3.text(b.get_x() + b.get_width() / 2,
             v + max(vals2) * 0.02,
             f'{v:.1f}W', ha='center', va='bottom', fontsize=8)
ax3.set_ylabel('Loss [W]')
ax3.set_title(f'Iron Losses\n(total {total_iron_W:.2f} W)')
ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(0, max(vals2) * 1.3)
ax3.tick_params(axis='x', labelsize=8)

# ---- (1,2): Key parameters text ----
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis('off')
params_text = (
    f'KEY PARAMETERS\n'
    f'─────────────────────\n'
    f'Qs = {Qs},  Qp = {Qp},  m = {m}\n'
    f'OD = {2*R_so:.0f} mm,  g = {g} mm\n'
    f'L  = {L_active*1000:.0f} mm\n'
    f'n  = {rpm} rpm\n'
    f'f_el = {fel:.0f} Hz\n'
    f'kw = {kw:.4f}\n'
    f'B_r = {B_r:.3f} T\n'
    f'B_g = {B_g_peak:.3f} T\n'
    f'\nFEM RESULTS\n'
    f'─────────────────────\n'
    f'T_mean = {T_mean:.1f} N·m\n'
    f'Ripple = {T_ripple:.1f}%\n'
    f'P_mech = {P_mech:.2f} kW\n'
    f'P_iron = {total_iron_W/1000:.3f} kW\n'
    f'\nIs = {Is:.0f} A (placeholder)'
)
ax4.text(0.05, 0.97, params_text, transform=ax4.transAxes,
         fontsize=8.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.savefig('results/summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/summary_dashboard.png')
print()
print('All cells complete. Report generation finished.')

Saved: results/summary_dashboard.png

All cells complete. Report generation finished.


/var/folders/66/l7z1xmd57kb2lqj_7cvg3c900000gp/T/ipykernel_11864/1558004606.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
